# 04-05. Four-box Model Summary  
# 4-box モデルのまとめ

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

---

## この Notebook の目的 / Goals

この Notebook は、04 シリーズのまとめです。  
This notebook summarizes the 04 series.

ここまでの流れは次の通りです。  
So far, the sequence was:

```text
04-01: Why Four-box?
       なぜ 4-box なのか

04-02: Reading and Building
       保存則とコードを読む

04-03: Parameter Experiments
       パラメタ感度実験

04-04: Modifying the Model
       モデルを自分で改造する
```

このまとめでは、細かいコードよりも、**何が分かったか**を整理します。  
In this summary, we focus less on code details and more on **what we learned**.

---

## 今日のゴール / Today's goals

1. 3-box から 4-box への拡張の意味を説明できる。  
   Explain the meaning of extending from 3-box to 4-box.

2. H, L, N, D の役割を説明できる。  
   Explain the roles of H, L, N, and D.

3. 重要なパラメタの意味を説明できる。  
   Explain the meaning of key parameters.

4. 感度実験の結果を研究報告風にまとめる。  
   Summarize sensitivity experiments in a research-report style.

5. 5〜8 分の短い発表資料に使える図と文章を作る。  
   Prepare figures and text for a short 5–8 minute presentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 3-box から 4-box へ / From 3-box to 4-box

3-box モデルでは、高緯度表層を 1 つの箱として扱いました。  
In the 3-box model, the high-latitude surface ocean was treated as one box.

```text
3-box:

H: High latitude
L: Low latitude
D: Deep ocean
```

しかし、現実の高緯度海洋には、少なくとも 2 つの重要な役割があります。  
However, high-latitude oceans have at least two important roles.

```text
North Atlantic:
  sinking / deep-water formation
  沈み込み・深層水形成

Southern Ocean:
  upwelling / air-sea CO2 exchange
  湧昇・大気海洋 CO2 交換
```

4-box モデルでは、この 2 つを分けます。  
The 4-box model separates these two roles.

```text
4-box:

H: Southern Ocean surface
L: Low-latitude surface
N: North Atlantic surface
D: Deep ocean
```

**重要な点 / Key point**

箱が 1 つ増えたこと自体が重要なのではありません。  
The important point is not simply that one box was added.

重要なのは、**沈み込みと湧昇を別々に扱えるようになったこと**です。  
The important point is that **sinking and upwelling can now be treated separately**.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

pos = {
    "H": (0.18, 0.55),
    "L": (0.50, 0.55),
    "N": (0.82, 0.55),
    "D": (0.50, 0.18),
}
labels = {
    "H": "H\nSouthern Ocean\nupwelling",
    "L": "L\nLow latitude\nbiology",
    "N": "N\nNorth Atlantic\nsinking",
    "D": "D\nDeep ocean\nstorage",
}

for b, (x, y) in pos.items():
    ax.text(x, y, labels[b], ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"))

ax.annotate("", xy=pos["H"], xytext=pos["D"], arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=pos["L"], xytext=pos["H"], arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=pos["N"], xytext=pos["L"], arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=pos["D"], xytext=pos["N"], arrowprops=dict(arrowstyle="->"))

ax.text(0.50, 0.82, "Atmosphere / 大気", ha="center",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black"))
ax.annotate("", xy=(0.18, 0.64), xytext=(0.40, 0.78), arrowprops=dict(arrowstyle="<->"))
ax.annotate("", xy=(0.50, 0.64), xytext=(0.50, 0.78), arrowprops=dict(arrowstyle="<->"))
ax.annotate("", xy=(0.82, 0.64), xytext=(0.60, 0.78), arrowprops=dict(arrowstyle="<->"))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Four-box model summary")
plt.show()

## 2. 各ボックスの役割 / Roles of the boxes

| Box | 日本語 | English | 重要な役割 / Main role |
|---|---|---|---|
| H | 南大洋表層 | Southern Ocean surface | 深層水の湧昇、大気 CO2 交換 |
| L | 低緯度表層 | Low-latitude surface | 暖かい表層、生物生産 |
| N | 北大西洋表層 | North Atlantic surface | 深層水形成、沈み込み |
| D | 深層 | Deep ocean | DIC・栄養塩の貯蔵、O2 消費 |

この 4 つの役割を覚えると、モデル式が読みやすくなります。  
Once you remember these four roles, the model equations become easier to read.

```text
H: deep water comes up
L: biology removes nutrients and carbon
N: surface water sinks
D: carbon and nutrients are stored
```

```text
H: 深層水が上がる
L: 生物が栄養塩と炭素を取り除く
N: 表層水が沈み込む
D: 炭素と栄養塩が蓄積する
```

## 3. 重要パラメタの整理 / Key parameters

4-box モデルで重要なパラメタは次です。  
The key parameters in the 4-box model are:

| Parameter | 日本語 | English | 主に効くもの |
|---|---|---|---|
| `Tcir` | 大規模循環 | large-scale circulation | N-D のつながり、深層ベンチレーション |
| `FHD` | H-D 交換 | H-D exchange | 南大洋深層ベンチレーション、大気 CO2 |
| `LF` | 低緯度生物ポンプ | low-latitude biological pump | L の輸出生産、表層 DIC |
| `CEPH` | 南大洋輸出生産 | Southern Ocean export | H の DIC/PO4、大気 CO2 |
| `RRC` | CaCO3 rain ratio | CaCO3 rain ratio | ALK, CO3, pCO2 |
| `air_sea` | 大気海洋 CO2 交換 | air-sea CO2 exchange | 大気 pCO2 |

**まとめ / Summary**

- `FHD` は、深層炭素が南大洋を通じて大気へ戻る経路に関係する。  
  `FHD` controls the pathway by which deep carbon returns to the atmosphere through the Southern Ocean.

- `CEPH` は、南大洋表層から深層へ炭素を輸送する強さに関係する。  
  `CEPH` controls carbon export from Southern Ocean surface to the deep ocean.

- `LF` は、広い低緯度海洋での生物ポンプに関係する。  
  `LF` controls biological export in the broad low-latitude ocean.

- `RRC` は、DIC だけでなく ALK も変える。  
  `RRC` changes not only DIC but also ALK.

## 4. まとめ用の軽量モデル / A compact model for summary figures

ここでは、まとめ用の軽量モデルを使って、発表用の図を作ります。  
Here we use a compact model to make figures for presentation.

この軽量モデルは、04-04 と同じく教育用の簡略版です。  
This compact model is a simplified teaching version, as in 04-04.

目的は、**図を作り、結果を説明する練習**です。  
The goal is to **make figures and practice explaining results**.

In [ ]:
CV1 = 1.0250e3
CV2 = 9.7561e-4
CV3 = 1.0e6
CV4 = 3.1536e7
CV5 = 8.64e4

def to_umolkg(x):
    return CV2 * 1.0e6 * x

def to_ppmv(x):
    return CV3 * x

VOCN_TOTAL = 1.292e18
AOCN = 3.49e14
VATM = 1.773e20

AREA = {"H": AOCN * 0.10, "L": AOCN * 0.75, "N": AOCN * 0.15}
VOLUME = {
    "H": AREA["H"] * 250.0,
    "L": AREA["L"] * 100.0,
    "N": AREA["N"] * 250.0,
}
VOLUME["D"] = VOCN_TOTAL - VOLUME["H"] - VOLUME["L"] - VOLUME["N"]

TEMP = {"H": 1.0, "L": 21.5, "N": 3.0, "D": 1.75}
DT = 8.64e4

RCP = 106.0
RNP = 16.0
RO2P = 172.0
RRC = 0.20
R = 1.0 / CV4
HSC = 2.0e-8 * CV1
DEL = 100.0

def carbonate_proxy(temp, alk, dic):
    dic_ref = 2.235e-3 * CV1
    alk_ref = 2.374e-3 * CV1
    temp_factor = np.exp(0.0423 * (temp - 15.0))
    pco2 = 280e-6 * (dic / dic_ref)**2 * (alk_ref / alk)**1.5 * temp_factor
    co3 = 220e-6 * CV1 * (alk / alk_ref) * (dic_ref / dic)
    return {"PCO2": pco2, "CO32": co3, "K0": 0.035}

def initial_state():
    x = {}
    for b in ["H", "L", "N", "D"]:
        x[f"PO4{b}"] = 2.10e-6 * CV1
        x[f"DIC{b}"] = 2.235e-3 * CV1
        x[f"ALK{b}"] = 2.374e-3 * CV1
        x[f"DO2{b}"] = 1.60e-4 * CV1
    x["PCO2A"] = 280e-6
    return x

def exports(x, LF=0.5, CEPH=0.75, CEPN=0.02):
    EPH = (CEPH / RCP) * AREA["H"] / CV4
    EPN = (CEPN / RCP) * AREA["N"] / CV4
    EPL = R * DEL * LF * x["PO4L"] * (x["PO4L"] / (HSC + x["PO4L"])) * VOLUME["L"]
    return {"H": EPH, "L": EPL, "N": EPN}

def step(x, Tcir=2.0e7, FHD=6.0e7, LF=0.5, CEPH=0.75, CEPN=0.02, RRC=0.20, gasH=1.0):
    y = dict(x)
    e = exports(x, LF=LF, CEPH=CEPH, CEPN=CEPN)
    EPH, EPL, EPN = e["H"], e["L"], e["N"]

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    gas = {}
    for b in ["H", "L", "N"]:
        c = carbonate_proxy(TEMP[b], x[f"ALK{b}"], x[f"DIC{b}"])
        scale = gasH if b == "H" else 1.0
        piston = 3.0 * AREA[b] / CV5 * scale
        gas[b] = piston * CV1 * c["K0"] * (x["PCO2A"] - c["PCO2"])

    def update(prefix, bio_factor, gas_terms=None):
        if gas_terms is None:
            gas_terms = {"H": 0.0, "L": 0.0, "N": 0.0}

        y[f"{prefix}H"] = x[f"{prefix}H"] + (
            Tcir * (x[f"{prefix}D"] - x[f"{prefix}H"])
            + FHD * (x[f"{prefix}D"] - x[f"{prefix}H"])
            - bio_factor * EPH
            + gas_terms.get("H", 0.0)
        ) * DT / VOLUME["H"]

        y[f"{prefix}L"] = x[f"{prefix}L"] + (
            Tcir * (x[f"{prefix}H"] - x[f"{prefix}L"])
            - bio_factor * EPL
            + gas_terms.get("L", 0.0)
        ) * DT / VOLUME["L"]

        y[f"{prefix}N"] = x[f"{prefix}N"] + (
            Tcir * (x[f"{prefix}L"] - x[f"{prefix}N"])
            - bio_factor * EPN
            + gas_terms.get("N", 0.0)
        ) * DT / VOLUME["N"]

        y[f"{prefix}D"] = x[f"{prefix}D"] + (
            Tcir * (x[f"{prefix}N"] - x[f"{prefix}D"])
            + FHD * (x[f"{prefix}H"] - x[f"{prefix}D"])
            + bio_factor * (EPH + EPL + EPN)
        ) * DT / VOLUME["D"]

    update("PO4", 1.0)
    update("ALK", alk_factor)
    update("DIC", dic_factor, gas_terms=gas)

    y["DO2H"] = x["DO2H"] + (RO2P * EPH) * DT / VOLUME["H"]
    y["DO2L"] = x["DO2L"] + (RO2P * EPL) * DT / VOLUME["L"]
    y["DO2N"] = x["DO2N"] + (RO2P * EPN) * DT / VOLUME["N"]
    y["DO2D"] = x["DO2D"] - (RO2P * (EPH + EPL + EPN)) * DT / VOLUME["D"]

    flux_to_atm = 0.0
    for b in ["H", "L", "N"]:
        c = carbonate_proxy(TEMP[b], y[f"ALK{b}"], y[f"DIC{b}"])
        scale = gasH if b == "H" else 1.0
        piston = 3.0 * AREA[b] / CV5 * scale
        flux_to_atm += piston * CV1 * c["K0"] * (c["PCO2"] - x["PCO2A"])
    y["PCO2A"] = x["PCO2A"] + flux_to_atm * DT / VATM

    return y, e

def diagnose(x, e=None):
    row = {}
    for b in ["H", "L", "N", "D"]:
        c = carbonate_proxy(TEMP[b], x[f"ALK{b}"], x[f"DIC{b}"])
        row[f"PO4{b}"] = to_umolkg(x[f"PO4{b}"])
        row[f"DIC{b}"] = to_umolkg(x[f"DIC{b}"])
        row[f"ALK{b}"] = to_umolkg(x[f"ALK{b}"])
        row[f"DO2{b}"] = to_umolkg(x[f"DO2{b}"])
        row[f"PCO2{b}"] = to_ppmv(c["PCO2"])
        row[f"CO3{b}"] = to_umolkg(c["CO32"])
    row["PCO2A"] = to_ppmv(x["PCO2A"])
    if e:
        row["EPH_PgCyr"] = e["H"] * RCP * 12e-15 * CV4
        row["EPL_PgCyr"] = e["L"] * RCP * 12e-15 * CV4
        row["EPN_PgCyr"] = e["N"] * RCP * 12e-15 * CV4
    return row

def run(years=1500, **kwargs):
    x = initial_state()
    hist = []
    for day in range(years * 365 + 1):
        if day % 365 == 0:
            _, e = step(x, **kwargs)
            row = {"year": day / 365}
            row.update(diagnose(x, e))
            hist.append(row)
        x, e = step(x, **kwargs)
    return x, pd.DataFrame(hist)

## 5. 発表用 Figure 1: 標準実験 / Presentation Figure 1: Standard experiment

まず標準実験の図を作ります。  
First, we make a figure for the standard experiment.

発表では、いきなり感度実験を見せる前に、標準状態を見せると分かりやすいです。  
In a presentation, it is helpful to show the standard case before sensitivity experiments.

In [ ]:
base_final, base = run()

plt.figure()
plt.plot(base["year"], base["PCO2A"], label="Atmosphere")
plt.plot(base["year"], base["PCO2H"], label="H")
plt.plot(base["year"], base["PCO2L"], label="L")
plt.plot(base["year"], base["PCO2N"], label="N")
plt.xlabel("Time [yr]")
plt.ylabel("pCO2 [ppmv]")
plt.title("Standard 4-box experiment")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
for b in ["H", "L", "N", "D"]:
    plt.plot(base["year"], base[f"PO4{b}"], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("PO4 [umol/kg]")
plt.title("Standard 4-box experiment: PO4")
plt.legend()
plt.grid(True)
plt.show()

### Figure caption / 図の説明文

**日本語**  
標準 4-box 実験では、時間とともに各ボックスの pCO2 と PO4 が分化する。これは、輸送、生物ポンプ、大気海洋 CO2 交換が各ボックスで異なる役割を持つためである。

**English**  
In the standard 4-box experiment, pCO2 and PO4 become different among boxes over time. This occurs because transport, biological export, and air-sea CO2 exchange play different roles in each box.

## 6. 発表用 Figure 2: 南大洋ベンチレーションの感度 / Presentation Figure 2: Southern Ocean ventilation sensitivity

4-box モデルの中心的な実験は、南大洋深層ベンチレーション `FHD` を変えることです。  
A central experiment in the 4-box model is changing Southern Ocean deep ventilation `FHD`.

`FHD` を弱めると、深層炭素が大気へ戻りにくくなります。  
Weakening `FHD` makes it harder for deep carbon to return to the atmosphere.

In [ ]:
cases_FHD = {
    "weak ventilation": {"FHD": 1.5e7},
    "standard": {"FHD": 6.0e7},
    "strong ventilation": {"FHD": 1.2e8},
}

summary_FHD = []
plt.figure()

for name, kwargs in cases_FHD.items():
    _, h = run(**kwargs)
    plt.plot(h["year"], h["PCO2A"], label=name)
    summary_FHD.append({
        "Case": name,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
        "Final DO2D": h["DO2D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Effect of Southern Ocean ventilation")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_FHD)

### Figure caption / 図の説明文

**日本語**  
南大洋深層ベンチレーションを弱めると、大気 pCO2 は低下する方向に変化する。この結果は、深層に蓄積した炭素が南大洋を通じて大気へ戻る経路が弱まるためと解釈できる。

**English**  
When Southern Ocean deep ventilation is weakened, atmospheric pCO2 tends to decrease. This can be interpreted as a weakening of the pathway by which carbon stored in the deep ocean returns to the atmosphere through the Southern Ocean.

## 7. 発表用 Figure 3: 生物ポンプの感度 / Presentation Figure 3: Biological pump sensitivity

次に、生物ポンプを変える実験です。  
Next, we examine biological pump sensitivity.

ここでは、低緯度生物ポンプ `LF` と南大洋生物ポンプ `CEPH` を比較します。  
Here we compare the low-latitude biological pump `LF` and the Southern Ocean biological pump `CEPH`.

In [ ]:
bio_cases = {
    "standard": {},
    "strong low-lat biology": {"LF": 1.5},
    "strong Southern Ocean biology": {"CEPH": 2.0},
    "strong both": {"LF": 1.5, "CEPH": 2.0},
}

summary_bio = []
plt.figure()

for name, kwargs in bio_cases.items():
    _, h = run(**kwargs)
    plt.plot(h["year"], h["PCO2A"], label=name)
    summary_bio.append({
        "Case": name,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final EPL": h["EPL_PgCyr"].iloc[-1],
        "Final EPH": h["EPH_PgCyr"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Effect of biological pump")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_bio)

### Figure caption / 図の説明文

**日本語**  
生物ポンプを強めると、表層から炭素が除去され、深層へ輸送されるため、大気 pCO2 は低下する方向に変化する。低緯度と南大洋では面積や深層とのつながりが異なるため、同じ生物ポンプでも効果は同じではない。

**English**  
Strengthening the biological pump removes carbon from the surface ocean and transfers it to the deep ocean, tending to lower atmospheric pCO2. The effects differ between the low-latitude ocean and the Southern Ocean because their area and connection to the deep ocean differ.

## 8. 発表用 Figure 4: 複合シナリオ / Presentation Figure 4: Combined scenario

実際の気候変化では、1 つの過程だけが変わるとは限りません。  
In real climate change, not only one process changes.

そこで、複数の変化を組み合わせたシナリオを作ります。  
Therefore, we create a combined scenario.

ここでは、氷期的な低 CO2 をイメージして、  
Here, imagining glacial low CO2, we combine:

```text
FHD ↓  : weaker Southern Ocean ventilation
CEPH ↑ : stronger Southern Ocean export
LF ↑   : stronger low-latitude export
gasH ↓ : weaker Southern Ocean gas exchange
```

を組み合わせます。

In [ ]:
scenario_cases = {
    "standard": {},
    "weak ventilation": {"FHD": 1.5e7},
    "strong biology": {"LF": 1.5, "CEPH": 2.0},
    "combined": {"FHD": 1.5e7, "LF": 1.5, "CEPH": 2.0, "gasH": 0.4},
}

summary_scenario = []
plt.figure()

for name, kwargs in scenario_cases.items():
    _, h = run(**kwargs)
    plt.plot(h["year"], h["PCO2A"], label=name)
    summary_scenario.append({
        "Case": name,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
        "Final DO2D": h["DO2D"].iloc[-1],
        "Final PO4H": h["PO4H"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Combined low-CO2 scenario")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_scenario)

### Figure caption / 図の説明文

**日本語**  
南大洋ベンチレーションの弱化と生物ポンプの強化を組み合わせると、大気 pCO2 は標準実験より大きく低下する。この結果は、複数の過程が同時に働くことで、単独実験より大きな応答が生じうることを示している。

**English**  
Combining weaker Southern Ocean ventilation with a stronger biological pump produces a larger decrease in atmospheric pCO2 than in the standard experiment. This shows that simultaneous changes in multiple processes can produce a larger response than single-factor experiments.

## 9. 研究報告風のまとめ / Research-style summary

ここまでの結果を、研究報告風にまとめます。  
Now we summarize the results in a research-report style.

### 背景 / Background

**日本語**  
3-box モデルでは高緯度海洋を 1 つの箱として扱っていたため、北大西洋の沈み込みと南大洋の湧昇を区別できなかった。4-box モデルでは、高緯度を北大西洋 N と南大洋 H に分けることで、深層水形成と南大洋ベンチレーションを別々に扱える。

**English**  
In the 3-box model, high-latitude oceans were represented as a single box, so North Atlantic sinking and Southern Ocean upwelling could not be distinguished. In the 4-box model, high latitudes are separated into the North Atlantic N box and Southern Ocean H box, allowing deep-water formation and Southern Ocean ventilation to be treated separately.

### 方法 / Method

**日本語**  
H, L, N, D の 4 つの箱からなる簡単な海洋炭素循環モデルを用いた。循環、深層ベンチレーション、生物ポンプ、大気海洋 CO2 交換を含め、各パラメタを変えた感度実験を行った。

**English**  
We used a simple ocean carbon-cycle model consisting of four boxes: H, L, N, and D. The model includes circulation, deep ventilation, biological export, and air-sea CO2 exchange. Sensitivity experiments were performed by changing key parameters.

### 結果 / Results

**日本語**  
南大洋深層ベンチレーションを弱めると、大気 pCO2 は低下する方向に変化した。また、生物ポンプを強めると表層から深層への炭素輸送が強まり、大気 pCO2 は低下した。複数の変化を組み合わせたシナリオでは、単独実験より大きな pCO2 低下が得られた。

**English**  
Weakening Southern Ocean deep ventilation tended to lower atmospheric pCO2. Strengthening the biological pump enhanced carbon transfer from the surface to the deep ocean and also lowered atmospheric pCO2. A combined scenario produced a larger pCO2 decrease than single-factor experiments.

### 解釈 / Interpretation

**日本語**  
4-box モデルでは、南大洋が深層炭素と大気を結ぶ重要な場所として働く。深層ベンチレーションが弱まると、深層に蓄積した炭素が大気へ戻りにくくなる。一方、生物ポンプが強まると、表層炭素が深層へ輸送されやすくなる。

**English**  
In the 4-box model, the Southern Ocean acts as an important connection between deep carbon and the atmosphere. When deep ventilation weakens, carbon stored in the deep ocean becomes less able to return to the atmosphere. When the biological pump strengthens, surface carbon is more efficiently transferred to the deep ocean.

## 10. 発表スライド構成案 / Suggested presentation structure

5〜8 分の短い発表なら、次の構成がよいです。  
For a short 5–8 minute presentation, the following structure works well.

### Slide 1: 目的 / Aim

**Title:**  
4-box モデルで海洋炭素循環を考える  
Thinking about ocean carbon cycling with a 4-box model

**Message:**  
3-box では区別できなかった北大西洋沈み込みと南大洋湧昇を、4-box では分けて扱える。

### Slide 2: モデル構造 / Model structure

図: H, L, N, D の模式図  
Figure: schematic of H, L, N, D

**Message:**  
H は南大洋、N は北大西洋を表す。

### Slide 3: 標準実験 / Standard experiment

図: pCO2 と PO4 の時系列  
Figure: time series of pCO2 and PO4

**Message:**  
輸送・生物ポンプ・ガス交換によって箱ごとの差が生まれる。

### Slide 4: 感度実験 / Sensitivity experiments

図: `FHD`, `LF`, `CEPH` の比較  
Figure: comparison of `FHD`, `LF`, `CEPH`

**Message:**  
南大洋ベンチレーションと生物ポンプは大気 CO2 に強く影響する。

### Slide 5: 複合シナリオ / Combined scenario

図: 標準実験と低 CO2 シナリオの比較  
Figure: standard vs low-CO2 scenario

**Message:**  
複数過程が同時に変わると、大気 CO2 の応答は大きくなる。

### Slide 6: まとめ / Summary

**Message:**  
4-box モデルにより、南大洋ベンチレーション・北大西洋沈み込み・生物ポンプの役割を分けて考えられる。

## 11. 最終課題 / Final exercises

### 課題 1 / Exercise 1

3-box から 4-box へ拡張することで、何が新しく表現できるようになったか。  
What becomes newly representable by extending from 3-box to 4-box?

### 課題 2 / Exercise 2

H box と N box の役割の違いを説明せよ。  
Explain the difference between the roles of the H box and the N box.

### 課題 3 / Exercise 3

`FHD` を弱めると、大気 pCO2 が低下しやすい理由を説明せよ。  
Explain why weakening `FHD` tends to lower atmospheric pCO2.

### 課題 4 / Exercise 4

生物ポンプを強めると、大気 pCO2 が低下しやすい理由を説明せよ。  
Explain why strengthening the biological pump tends to lower atmospheric pCO2.

### 課題 5 / Exercise 5

4-box モデルの限界を 3 つ挙げよ。  
List three limitations of the 4-box model.

### 課題 6 / Exercise 6

短い研究報告として、標準実験と複合シナリオの違いを 5〜10 行で説明せよ。  
As a short research report, explain the difference between the standard experiment and the combined scenario in 5–10 lines.

## 12. 次へ / Next step

これで 04 Four-box シリーズは一区切りです。  
This completes the 04 Four-box series.

次の 05 では、さらに箱を増やして、海洋内部構造をより現実に近づけます。  
In 05, we increase the number of boxes further and make the internal ocean structure more realistic.

```text
04: Four-box
    H, L, N, D

05: Six-box
    surface + intermediate/deep structure
```

4-box で学んだ大事な考えは、次でも同じです。  
The important idea learned in the 4-box model remains the same:

```text
なぜ箱を増やすのか？
↓
新しい科学的問いを扱うため
```

```text
Why increase the number of boxes?
↓
To ask new scientific questions
```